# 3.5 RAG Evaluation

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Learning objectives

- Build a golden eval set (question, ground-truth answer, expected source chunks) for a RAG pipeline.
- Run the four core RAGAS metrics and map each low score to a specific pipeline failure.
- Measure retrieval alone (hit rate at k) separately from end-to-end quality, and know when each matters.
- Build a small structured-output LLM judge with an explicit rubric and calibrate it against your own labels.
- Price an eval run, schedule it sensibly, and turn the eval set into a regression test that gates changes.

## Eyeballing does not scale

By lesson 3.4 you have a working RAG pipeline. You tested it on three questions you made up, it answered all three well, and it feels done. This is the exact moment RAG pipelines get shipped and then embarrass people in production. The reason is boring and statistical: you tested the head of the query distribution, and RAG fails in the tail. Your three test questions were the kind you would ask. Real users ask things with typos, wrong product names, missing context, and multi-hop questions. Your chunking looks fine on the pages you read and breaks on the ones you did not.

Evaluation replaces "it feels fine" with numbers you can track over time. The habit to build here is the same one you built in Module 2 around traces: never judge a pipeline by the answer alone; always look at what went into the answer. For RAG, that means scoring the retrieval and the generation separately, because they fail separately, and the fix for each is different.

## The golden set

Every serious eval starts with a golden set: a list of questions with known-correct answers and known-correct sources. The minimum schema per row:

```json
{
  "question": "How long does the free trial last?",
  "reference_answer": "30 days from the first sync.",
  "expected_chunk_ids": ["billing-plans-003", "billing-plans-004"]
}
```

Three fields, each with a job:

- **question**: what the user asked, in the user's own words.
- **reference_answer**: the correct answer, written or checked by a human. Not copied from your pipeline's output; if your pipeline is wrong and your reference comes from your pipeline, every metric congratulates you on being wrong consistently.
- **expected_chunk_ids**: which chunks the retriever *should* return. This needs your chunks to carry stable ids (a doc name plus an index works). Without this field you can only evaluate end to end, which tells you something broke but not where.

Where the questions come from matters more than how many there are. Start with 15 to 30. The best questions are real ones: logs from a beta, support tickets, questions colleagues ask about the docs. A good golden set also includes deliberate edge cases: a question the docs cannot answer (the pipeline must abstain, lesson 3.4), a question with a synonym the embedding model might miss ("how much does it cost" vs "pricing"), and a question that needs facts from two different pages (the single-chunk retrieval case will fail it, and you want to see that in the numbers).

A GDPR note, because your audience lives in Europe and so do their users. A golden set built from real user queries is personal data: queries contain names, emails, company details, and sometimes health or financial information. Article 5 of the GDPR requires data minimisation, so strip PII before the set goes into your eval harness, keep the set on your own machine rather than in a shared repo, and if you run it through a hosted eval service, check that a data processing agreement covers it. Synthetic questions you write yourself carry none of this baggage; real ones do.

## The four metrics that partition failure

RAGAS ships metric classes for RAG. The four core ones split the pipeline's failure modes between retrieval and generation, which is exactly the split you need for diagnosis:

| Metric | Compares | Low score means |
|---|---|---|
| Faithfulness | answer vs retrieved chunks | the generator invented or distorted facts (a hallucination problem; fix grounding, lesson 3.4) |
| Answer relevancy | answer vs question | the answer wanders or dodges (a prompt or retrieval-noise problem) |
| Context precision | retrieved chunks vs reference answer | the retriever returns junk at the top of the ranking (fix retrieval, lesson 3.3) |
| Context recall | retrieved chunks vs reference answer | the retriever misses needed information (fix coverage: chunking, k, hybrid search) |

Two of them need the reference answer (precision and recall); faithfulness and answer relevancy do not, which means you can measure grounding even where you have no labelled answer. That is a useful property: it lets you score production logs retroactively.

## Running RAGAS the current way

RAGAS 0.4.x uses a collections-based metric API. The old style you will see in 2025 tutorials (`from ragas.metrics import faithfulness, answer_relevancy` and the bare `evaluate()` function) is deprecated and slated for removal in RAGAS 1.0, so learn the current one. Metric classes take a wrapped judge LLM (and embeddings where the metric needs them), and each metric scores one sample object:

In [ ]:
# pip install ragas==0.4.3
import asyncio
from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

judge = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
judge_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

metrics = [
    Faithfulness(llm=judge),                              # answer vs chunks
    AnswerRelevancy(llm=judge, embeddings=judge_embeddings),  # answer vs question
    ContextPrecision(llm=judge),                          # chunk quality vs reference
    ContextRecall(llm=judge),                             # chunk coverage vs reference
]

sample = SingleTurnSample(
    user_input="How long does the free trial last?",
    response="The free trial lasts 30 days from your first sync. [1]",
    retrieved_contexts=[
        "Trial: one click on the Plans page starts a 30-day trial, counted from the first sync.",
        "Paid plans are billed through the Shopify Billing API.",
    ],
    reference="30 days from the first sync.",
)

async def score_all(sample):
    rows = {}
    for metric in metrics:
        result = await metric.single_turn_ascore(sample)
        rows[type(metric).__name__] = getattr(result, "value", result)
    return rows

print(asyncio.run(score_all(sample)))
# {'Faithfulness': 1.0, 'AnswerRelevancy': 0.93, 'ContextPrecision': 0.5, 'ContextRecall': 1.0}

Read that toy result like a diagnosis: faithfulness 1.0 and recall 1.0 mean the answer is grounded and the needed chunk was retrieved, but context precision 0.5 says half the retrieved context was irrelevant (the billing chunk added nothing). The fix is retrieval hygiene, not generation: smaller k, a reranker, or MMR from lesson 3.3. That sentence is the whole point of the four-metric split. One number never told you that.

For a full golden set, loop over your rows with `asyncio`, gather scores, and dump them into a pandas DataFrame: per-row scores are what you drill into, column means are what you graph. RAGAS also offers an `@experiment` decorator for running a pipeline and metrics together, which is the right tool once the scoring loop gets repetitive; start with the explicit loop, because explicit shows you where the cost comes from.

## An LLM-as-judge you built yourself

RAGAS metrics are themselves LLM judges under the hood. Knowing how to write one by hand matters, because sooner or later a canned metric will not match your quality bar and you will write a custom judge. The recipe is three ingredients: a rubric with numbered levels, a structured-output schema so the verdict is machine-readable, and temperature 0 for repeatability.

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

class GroundingVerdict(BaseModel):
    claim: str = Field(description="One factual claim extracted from the answer.")
    supported: bool = Field(
        description="True only if the claim follows from the retrieved context."
    )
    evidence: str = Field(
        description="A short quote from the context that settles it, or 'no evidence'."
    )

judge = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(
    GroundingVerdict
)

rubric = (
    "You are checking whether an answer is grounded in its retrieved context. "
    "Split the answer into individual factual claims. For ONE claim, reply with: "
    "the claim in the 'claim' field; 'supported' true only if the claim is stated "
    "or logically follows from the context; and a short verbatim quote in "
    "'evidence'. Boil 'the context mentions X' down to exact wording. "
    "If the claim goes beyond the context in any detail, supported is false."
)

verdict = judge.invoke(
    f"{rubric}\n\nContext: Trial lasts 30 days from the first sync.\n"
    "Answer: The trial lasts 30 days and includes phone support."
)
print(verdict.model_dump())
# {'claim': 'The trial includes phone support.', 'supported': False, 'evidence': 'no evidence'}

Note what this judge does differently from faithfulness in RAGAS: it works claim by claim with quoted evidence, which makes failures debuggable. A single score tells you something broke; a claim list tells you which claim broke and what the context actually said. When you write custom judges, design the output for debugging, not for dashboards.

## Retrieval-only eval first

The cheapest, fastest eval you can run needs no LLM at all. If your golden set has `expected_chunk_ids`, you can measure retrieval directly: for each question, run the retriever, check whether the expected chunks appear in the top k, and report the fraction. This is hit rate at k, and it runs in seconds.

In [ ]:
def hit_rate_at_k(golden_rows, retriever, k=5):
    hits = 0
    for row in golden_rows:
        retrieved = retriever.invoke(row["question"])[:k]
        got_ids = {d.metadata["chunk_id"] for d in retrieved}
        if set(row["expected_chunk_ids"]) & got_ids:
            hits += 1
    return hits / len(golden_rows)

Run this before you ever touch an LLM metric, and re-run it after every retrieval change: a new chunk size, a new k, hybrid search, a reranker. It is deterministic, free, and instant, which means you will actually run it. The pattern from lesson 3.1 (evaluating chunking) belongs here as a permanent habit: change the chunking, run the hit-rate script, keep the config only if the number goes up. End-to-end LLM metrics are for the weekly review; retrieval-only checks are for the daily loop.

## Evals as regression tests

A golden set that sits in a notebook is a souvenir. Wire it into your workflow so it runs whenever the pipeline changes. The simplest useful form is a threshold gate in CI: score the golden set, assert each metric is above its floor, fail the build otherwise.

In [ ]:
scores = asyncio.run(score_golden_set())
assert scores["Faithfulness"] >= 0.85, scores
assert scores["ContextRecall"] >= 0.70, scores

Two pieces of craft make this work instead of annoying everyone. First, set thresholds from your first measured baseline, not from round numbers: run the set once, see 0.81 faithfulness, set the gate at 0.78. Gates should catch regressions, not certify excellence. Second, keep the golden set small enough that the gate runs in minutes, and keep a larger set for the weekly deep run. A gate that takes an hour and costs real money per commit gets disabled within a week.

## Eval cost math

Evals multiply model calls, and the multiplication surprises people who are used to thinking in per-query costs. Work it out once on paper so the bill never surprises you:

- 1 golden sample through 4 RAGAS metrics is roughly 5 judge calls (faithfulness, relevancy plus embeddings, precision, recall) plus the pipeline's own calls (embed query, retrieve, generate).
- 20 samples is then on the order of 100 judge calls per eval run.
- On a small judge model that is cheap in money and minutes; on a frontier model it is neither.

Three cost rules follow. First, the judge model does not have to be the generator model; a small model with a good rubric usually agrees with the big one on grounding questions, and you should verify that agreement once (the calibration section below) rather than assuming it. Second, run the full LLM-judged suite on a schedule, nightly or per release, not on every code tweak; the retrieval-only hit-rate check is your inner loop. Third, watch the calendar cost as well as the money cost: a team that only runs evals when they remember has no regression testing; the cron job is part of the deliverable.

## Flaky judges and how to calibrate them

LLM judges are noisy instruments. Developers' most common complaints about RAG evals are all about this: scores change between runs, a judge rewards long answers, a judge agrees with itself less than you would like. None of this makes judges useless, but it does mean you treat them like a measuring device that needs calibration, not like a unit test.

- **Fix the temperature.** temperature=0, always, for judges. A judge that samples creatively is a random number generator with opinions.
- **Write the rubric in words, not vibes.** "supported" has to be defined (your hand-built judge above does this). Vague rubrics produce vague scores.
- **Measure agreement, do not assume it.** Score each sample twice and compute the disagreement rate; if the judge disagrees with itself on more than a few percent of samples, the rubric or the model is wrong, not the pipeline. For high-stakes gates, use two different judge models and require agreement.
- **Calibrate against humans.** Take 20 samples, score them yourself claim by claim, and compare with the judge. Where they diverge, fix the rubric, not the pipeline. This one afternoon of calibration is what turns the judge from a toy into an instrument, and it is the step most teams skip.
- **Know the biases.** Judges favour longer answers, favour the answer shown first in pairwise comparisons, and mildly favour their own model's outputs. Mitigations: rubric-based absolute scoring instead of pairwise comparisons, and a judge model from a different family than the generator when self-preference would matter.

## RAGAS, LangSmith, or a hand-rolled judge

Learners ask which tool to adopt. There is no single right answer, but there is a clear decision path for where you are now:

- **RAGAS** is the default for this module. No account, standard metric definitions the whole industry recognises, costs only the judge API calls. Best fit: offline batch evals of a RAG pipeline, exactly what you are building.
- **Hand-rolled judges** (the claim-by-claim pattern above) are for the metric RAGAS does not ship: your domain's specific quality bar, citation correctness, tone rules. Expect to write at least one per serious project.
- **LangSmith evaluation** is for when you already trace in LangSmith (Module 6). It keeps datasets, runs, and evaluators in one hosted place: `from langsmith.evaluation import evaluate`, then `evaluate(target_fn, data="my_dataset", evaluators=[my_evaluator], experiment_prefix="...")`, where evaluators are plain functions taking `(inputs, outputs, reference_outputs)`. It needs an account and sends your data to a hosted service, so the GDPR note above applies doubly.
- **DeepEval** (pytest-style, `deepeval test run`) is worth a mention as the CI-native option if your team already lives in pytest; it is optional for this course.

Start with RAGAS for the standard metrics and one hand-rolled judge for your custom bar. Graduate to LangSmith when tracing is already part of your workflow.

## What can go wrong

- **Reference answers copied from your own pipeline.** The metrics then measure self-consistency, not quality. Write references by hand or from the source docs, never from pipeline output.
- **Golden sets that only cover the head.** Ten easy questions give a perfect score and zero information. Every set needs abstention probes, synonym traps, and at least one multi-hop question.
- **Optimising for the metric instead of the user.** Answer relevancy can be gamed by confident-sounding non-answers; faithfulness can be gamed by quoting chunks verbatim without answering. Metrics are instruments, not the goal. The calibration section is the antidote.
- **Judge model drift.** The provider updates gpt-4o-mini and your historical scores shift without any pipeline change. Record the judge model name and version with every run, same as you pin library versions.
- **Evals that cost more than the feature.** A 200-sample set with 4 metrics on a frontier judge, run per commit, is a budget line item. Keep the CI gate small, the weekly set large, and the judge model small.
- **Flaky gates that everyone ignores.** A gate that fails on judge noise trains the team to re-run until it passes, which trains them to ignore real regressions. Calibrate first, set thresholds from measured baselines, and investigate every failure before re-running.
- **PII in golden sets.** Real user queries in a shared repo or a hosted eval service without a data processing agreement is a GDPR problem waiting to happen. Scrub, minimise, keep local.
- **Version skew between lesson and library.** RAGAS moves fast: the legacy lowercase metric functions and the old `evaluate()` are deprecated in 0.4.x and will be removed in 1.0. Pin `ragas==0.4.3` in your requirements, and if a tutorial's imports fail, check whether it is teaching the legacy API.

## Check your understanding

**1. Your golden set has questions and answers but no expected chunk ids. What can you evaluate, and what have you lost?**
You can evaluate end to end: faithfulness and answer relevancy need no chunk labels. You have lost the ability to separate retrieval failures from generation failures. When context precision drops you will see it in the aggregate numbers, but you cannot say "the retriever missed chunk X" versus "the generator ignored chunk X", and you cannot run the free retrieval-only hit-rate check.

**2. Context precision is 0.4 and faithfulness is 0.95. What is broken and where do you fix it?**
Retrieval is returning mostly irrelevant chunks (precision 0.4) while the generator is faithfully using what it got (faithfulness 0.95). The pipeline is honestly answering from bad evidence. Fix retrieval: rerank, smaller k, MMR or hybrid search from lesson 3.3, or revisit chunking. Touching the generator or the prompt would be fixing the wrong layer.

**3. Why should reference answers never come from your own pipeline's output?**
Because then the metrics measure agreement with the pipeline rather than correctness. A pipeline that hallucinates consistently will score perfectly against references it generated itself. References must come from an independent source: the docs, a human, or a different system.

**4. An eval gate in CI fails intermittently with no pipeline change. Name two causes and how you would confirm each.**
Judge noise: re-run the same samples with temperature=0 and measure self-disagreement; if scores move, the judge is the cause. Judge model drift: check whether the provider changed the judge model version between runs; recorded judge versions in the run log confirm it. A third possibility is retrieval nondeterminism (approximate nearest neighbours), confirmed by running the retriever twice on the same query.

**5. A golden set built from real support tickets is about to be committed to a shared repo. What is the GDPR problem and what do you do instead?**
The tickets are personal data: names, emails, possibly sensitive details. Committing them exposes personal data to everyone with repo access with no lawful basis and no minimisation, which breaches GDPR. Scrub PII from every query first, keep the set local or in access-controlled storage, set a retention limit, and never run it through a hosted eval service without a data processing agreement.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.